In [ ]:
# ruff: noqa F401

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from utils import data_path, Set1

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

from sklearn.decomposition import KernelPCA

# Kernel PCA

[Kernel PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.KernelPCA.html#sklearn.decomposition.KernelPCA) and [Example](https://scikit-learn.org/stable/modules/decomposition.html#kernel-pca)

In [ ]:
def smiles_to_bits(smiles):
    """Convert a SMILES string to bits."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return morgan_gen.GetFingerprintAsNumPy(mol).astype(np.uint8)


def array_to_bitvect(x):
    """Convert a bit array to RDKit format."""
    # x is a 1D array of 0/1 values
    bitstring = "".join(x.astype(str))
    return DataStructs.CreateFromBitString(bitstring)


def tversky_kernel(x, y, alpha=0.7, beta=0.3):
    """Compute Tversky similarity for two fingerprints."""
    fp_x = array_to_bitvect(x)
    fp_y = array_to_bitvect(y)
    return DataStructs.TverskySimilarity(fp_x, fp_y, alpha, beta)

In [ ]:
data = data_path("kernel_pca", "psychedelics_molecules.csv")
df = pd.read_csv(data, index_col=0)
display(df)

In [ ]:
# -------------------------
# 1) SMILES -> bit vectors
# -------------------------
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,  # ECFP4
    fpSize=1024,
)

# df["smiles"] should contain SMILES strings
fps = df["Canonical SMILES"].apply(smiles_to_bits)
mask = fps.notna()

X = np.vstack(fps[mask].to_list())  # shape: (n_samples, n_bits)
df_valid = df.loc[mask].reset_index(drop=True)

In [ ]:
# -----------------------------
# 4) Kernel PCA
# -----------------------------
kpca = KernelPCA(
    n_components=10,  # choose however many you want
    kernel=lambda a, b: tversky_kernel(a, b, alpha=0.7, beta=0.3),
    eigen_solver="auto",
)

Z = kpca.fit_transform(X)

In [ ]:
# -----------------------------
# 5) Scatter plot of first two components
# -----------------------------
plt.figure(figsize=(8, 6))

categories = df_valid["Category"].str.strip().astype(str)
unique_cats = sorted(categories.unique())
colors = Set1[min(len(unique_cats), 9)]

for i, cat in enumerate(unique_cats):
    idx = categories == cat
    plt.scatter(
        Z[idx, 0],
        Z[idx, 1],
        s=35,
        alpha=0.8,
        label=cat,
        c=colors[i % len(colors)],
    )

plt.xlabel("Kernel PC1")
plt.ylabel("Kernel PC2")
plt.title("Kernel PCA with RDKit Tversky kernel")
plt.legend(title="Category", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

# -----------------------------
# 6) Scree plot
# -----------------------------
eigvals = kpca.eigenvalues_

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(eigvals) + 1), eigvals, marker="o")
plt.xlabel("Component")
plt.ylabel("Eigenvalue")
plt.title("Kernel PCA scree plot")
plt.xticks(range(1, len(eigvals) + 1))
plt.tight_layout()
plt.show()